In [10]:
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
          "GB")
else:
    print("Running on CPU")

PyTorch version: 2.14.1+cu130
GPU available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [11]:
!pip install -q transformers datasets peft accelerate bitsandbytes sentencepiece

In [12]:
import torch
import transformers
import datasets
import peft
import accelerate
import bitsandbytes

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)
print("Bitsandbytes:", bitsandbytes.__version__)
print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.14.1+cu130
Transformers: 5.22.1
Datasets: 4.8.5
PEFT: 0.21.2
Accelerate: 1.15.0
Bitsandbytes: 0.50.2
GPU: Tesla T4


In [13]:
import random
import json

random.seed(42)

products = ["laptop", "smartphone", "headphones", "smartwatch"]
issues = ["battery", "screen", "connectivity", "audio", "charging"]
urgencies = ["low", "medium", "high"]
actions = ["repair", "replacement", "refund", "troubleshooting"]

templates = [
    "My {product} has a {issue} problem. This is {urgency} priority. Please arrange a {action}.",
    "I need help with my {product}. The issue is {issue}, urgency is {urgency}. I would like {action}.",
    "Please help me with a {issue} issue on my {product}. Mark it as {urgency} and request {action}.",
    "For my {product}, I am reporting {issue}. It is a {urgency} priority case. My preferred solution is {action}.",
    "The {product} is experiencing {issue}. I consider this {urgency} urgency and want a {action}.",
    "I have a {urgency} priority complaint about the {issue} on my {product}. Please provide {action}.",
    "Regarding the {product}, the problem concerns {issue}. My requested resolution is {action}; the urgency is {urgency}.",
    "Could you classify this support request? Product: {product}. Problem: {issue}. Priority: {urgency}. Desired action: {action}."
]

def create_example(template):
    product = random.choice(products)
    issue = random.choice(issues)
    urgency = random.choice(urgencies)
    action = random.choice(actions)

    message = template.format(
        product=product,
        issue=issue,
        urgency=urgency,
        action=action
    )

    output = {
        "product": product,
        "issue_type": issue,
        "urgency": urgency,
        "requested_action": action
    }

    return {
        "input": message,
        "output": json.dumps(output, separators=(",", ":"))
    }

train_data = [create_example(t) for t in templates[:6] for _ in range(200 // 6)]
train_data += [create_example(templates[0]) for _ in range(200 - len(train_data))]

val_data = [create_example(templates[6]) for _ in range(50)]
test_data = [create_example(templates[7]) for _ in range(50)]

random.shuffle(train_data)

print("Training examples:", len(train_data))
print("Validation examples:", len(val_data))
print("Test examples:", len(test_data))

print("\nSample training example:")
print("Input:", train_data[0]["input"])
print("Expected output:", train_data[0]["output"])

print("\nSample test example:")
print("Input:", test_data[0]["input"])
print("Expected output:", test_data[0]["output"])

Training examples: 200
Validation examples: 50
Test examples: 50

Sample training example:
Input: I have a low priority complaint about the battery on my headphones. Please provide replacement.
Expected output: {"product":"headphones","issue_type":"battery","urgency":"low","requested_action":"replacement"}

Sample test example:
Input: Could you classify this support request? Product: smartphone. Problem: screen. Priority: low. Desired action: repair.
Expected output: {"product":"smartphone","issue_type":"screen","urgency":"low","requested_action":"repair"}


In [14]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Model loaded:", model_name)
print("Model device:", base_model.device)
print("GPU memory allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded: Qwen/Qwen2.5-0.5B-Instruct
Model device: cuda:0
GPU memory allocated: 1.85 GB


In [15]:
import json
import re
import torch
from tqdm.auto import tqdm

base_model.eval()

def generate_answer(model, message):
    messages = [
        {
            "role": "system",
            "content": (
                "Extract the customer support information. "
                "Return only a valid JSON object with these keys: "
                "product, issue_type, urgency, requested_action."
            )
        },
        {
            "role": "user",
            "content": message
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()


def extract_json(text):
    match = re.search(r"\{.*?\}", text, re.DOTALL)

    if not match:
        return None

    try:
        return json.loads(match.group())
    except json.JSONDecodeError:
        return None


valid_json = 0
exact_matches = 0
baseline_predictions = []

for example in tqdm(test_data):
    prediction = generate_answer(base_model, example["input"])
    parsed = extract_json(prediction)
    expected = json.loads(example["output"])

    if parsed is not None:
        valid_json += 1

    if parsed == expected:
        exact_matches += 1

    baseline_predictions.append({
        "input": example["input"],
        "expected": expected,
        "generated": prediction,
        "parsed": parsed
    })

print("Total test examples:", len(test_data))
print("Valid JSON:", valid_json)
print("JSON validity rate:", round(valid_json / 50 * 100, 2), "%")
print("Exact matches:", exact_matches)
print("Exact-match accuracy:", round(exact_matches / 50 * 100, 2), "%")

  0%|          | 0/50 [00:00<?, ?it/s]

Total test examples: 50
Valid JSON: 50
JSON validity rate: 100.0 %
Exact matches: 35
Exact-match accuracy: 70.0 %


In [16]:
failures = [
    item for item in baseline_predictions
    if item["parsed"] != item["expected"]
]

print("Number of incorrect predictions:", len(failures))

for i, item in enumerate(failures[:5], 1):
    print(f"\nFailure {i}")
    print("Input:", item["input"])
    print("Expected:", item["expected"])
    print("Predicted:", item["parsed"])

Number of incorrect predictions: 15

Failure 1
Input: Could you classify this support request? Product: headphones. Problem: battery. Priority: high. Desired action: repair.
Expected: {'product': 'headphones', 'issue_type': 'battery', 'urgency': 'high', 'requested_action': 'repair'}
Predicted: {'product': 'headphones', 'issue_type': 'battery issue', 'urgency': 'high', 'requested_action': 'repair'}

Failure 2
Input: Could you classify this support request? Product: smartphone. Problem: battery. Priority: high. Desired action: repair.
Expected: {'product': 'smartphone', 'issue_type': 'battery', 'urgency': 'high', 'requested_action': 'repair'}
Predicted: {'product': 'smartphone', 'issue_type': 'battery issue', 'urgency': 'high', 'requested_action': 'repair'}

Failure 3
Input: Could you classify this support request? Product: smartphone. Problem: battery. Priority: medium. Desired action: refund.
Expected: {'product': 'smartphone', 'issue_type': 'battery', 'urgency': 'medium', 'requested_a

In [17]:
import traceback

try:
    model = get_peft_model(base_model, lora_config)
    model.print_trainable_parameters()
except Exception:
    traceback.print_exc()

Traceback (most recent call last):
  File "/tmp/ipykernel_1609/1921611559.py", line 4, in <cell line: 0>
    model = get_peft_model(base_model, lora_config)
  File "/usr/local/lib/python3.13/dist-packages/peft/mapping_func.py", line 198, in get_peft_model
    return MODEL_TYPE_TO_PEFT_MODEL_MAPPING[peft_config.task_type](
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        model,
        ^^^^^^
    ...<3 lines>...
        low_cpu_mem_usage=low_cpu_mem_usage,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/peft/peft_model.py", line 2082, in __init__
    super().__init__(model, peft_config, adapter_name, **kwargs)
    ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/peft/peft_model.py", line 177, in __init__
    self.base_model = cls(model, {adapter_name: peft_config}, adapter_name)
                      ~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

In [18]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [19]:
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


In [20]:
def format_example(example):
    messages = [
        {
            "role": "system",
            "content": (
                "Extract customer support information. "
                "Return only valid JSON with these keys: "
                "product, issue_type, urgency, requested_action."
            )
        },
        {
            "role": "user",
            "content": example["input"]
        },
        {
            "role": "assistant",
            "content": example["output"]
        }
    ]

    return {
        "text": tokenizer.apply_chat_template(
            messages,
            tokenize=False
        )
    }

from datasets import Dataset

train_dataset = Dataset.from_list(train_data).map(format_example)
val_dataset = Dataset.from_list(val_data).map(format_example)

print("Training rows:", len(train_dataset))
print("Validation rows:", len(val_dataset))
print("\nFormatted training example:\n")
print(train_dataset[0]["text"])

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Training rows: 200
Validation rows: 50

Formatted training example:

<|im_start|>system
Extract customer support information. Return only valid JSON with these keys: product, issue_type, urgency, requested_action.<|im_end|>
<|im_start|>user
I have a low priority complaint about the battery on my headphones. Please provide replacement.<|im_end|>
<|im_start|>assistant
{"product":"headphones","issue_type":"battery","urgency":"low","requested_action":"replacement"}<|im_end|>



In [21]:
def tokenize_example(example):
    prompt_messages = [
        {
            "role": "system",
            "content": (
                "Extract customer support information. "
                "Return only valid JSON with these keys: "
                "product, issue_type, urgency, requested_action."
            )
        },
        {
            "role": "user",
            "content": example["input"]
        }
    ]

    prompt = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True
    )

    full_text = tokenizer.apply_chat_template(
        [
            *prompt_messages,
            {"role": "assistant", "content": example["output"]}
        ],
        tokenize=False
    )

    prompt_ids = tokenizer(
        prompt,
        add_special_tokens=False
    )["input_ids"]

    full_ids = tokenizer(
        full_text,
        add_special_tokens=False,
        truncation=True,
        max_length=512
    )["input_ids"]

    labels = full_ids.copy()
    labels[:len(prompt_ids)] = [-100] * min(
        len(prompt_ids), len(labels)
    )

    return {
        "input_ids": full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels": labels
    }


tokenized_train = Dataset.from_list(train_data).map(
    tokenize_example,
    remove_columns=Dataset.from_list(train_data).column_names
)

tokenized_val = Dataset.from_list(val_data).map(
    tokenize_example,
    remove_columns=Dataset.from_list(val_data).column_names
)

print("Tokenized training examples:", len(tokenized_train))
print("Tokenized validation examples:", len(tokenized_val))
print("First example token count:", len(tokenized_train[0]["input_ids"]))
print("First example supervised tokens:",
      sum(x != -100 for x in tokenized_train[0]["labels"]))

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Tokenized training examples: 200
Tokenized validation examples: 50
First example token count: 76
First example supervised tokens: 23


In [22]:
from transformers import (
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq
)

training_args = TrainingArguments(
    output_dir="./qwen-lora-results",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    optim="adamw_torch",
    seed=42
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=-100,
    pad_to_multiple_of=8
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    processing_class=tokenizer
)

print("Trainer configured successfully.")
print("Training examples:", len(tokenized_train))
print("Validation examples:", len(tokenized_val))

Trainer configured successfully.
Training examples: 200
Validation examples: 50


In [23]:
train_result = trainer.train()

print("\nTraining completed!")
print("Training loss:", train_result.training_loss)
print("Training runtime (seconds):",
      train_result.metrics["train_runtime"])
print("Training samples per second:",
      train_result.metrics["train_samples_per_second"])

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss
1,0.071227,0.004285
2,0.004017,0.002961
3,0.003075,0.002536



Training completed!
Training loss: 0.08118247448538359
Training runtime (seconds): 23.1788
Training samples per second: 25.886


In [24]:
model.eval()

lora_valid_json = 0
lora_exact_matches = 0
lora_predictions = []

for example in tqdm(test_data):
    prediction = generate_answer(model, example["input"])
    parsed = extract_json(prediction)
    expected = json.loads(example["output"])

    if parsed is not None:
        lora_valid_json += 1

    if parsed == expected:
        lora_exact_matches += 1

    lora_predictions.append({
        "input": example["input"],
        "expected": expected,
        "generated": prediction,
        "parsed": parsed
    })

print("Total test examples:", len(test_data))
print("Valid JSON:", lora_valid_json)
print("JSON validity rate:",
      round(lora_valid_json / 50 * 100, 2), "%")
print("Exact matches:", lora_exact_matches)
print("Exact-match accuracy:",
      round(lora_exact_matches / 50 * 100, 2), "%")

  0%|          | 0/50 [00:00<?, ?it/s]

Total test examples: 50
Valid JSON: 50
JSON validity rate: 100.0 %
Exact matches: 50
Exact-match accuracy: 100.0 %


In [25]:
for i, item in enumerate(failures[:3], 1):
    print(f"\n{'='*50}")
    print(f"Example {i}")
    print("Input:", item["input"])
    print("Expected:", item["expected"])
    print("Base model:", item["parsed"])

    lora_item = next(
        p for p in lora_predictions
        if p["input"] == item["input"]
    )
    print("LoRA model:", lora_item["parsed"])


Example 1
Input: Could you classify this support request? Product: headphones. Problem: battery. Priority: high. Desired action: repair.
Expected: {'product': 'headphones', 'issue_type': 'battery', 'urgency': 'high', 'requested_action': 'repair'}
Base model: {'product': 'headphones', 'issue_type': 'battery issue', 'urgency': 'high', 'requested_action': 'repair'}
LoRA model: {'product': 'headphones', 'issue_type': 'battery', 'urgency': 'high', 'requested_action': 'repair'}

Example 2
Input: Could you classify this support request? Product: smartphone. Problem: battery. Priority: high. Desired action: repair.
Expected: {'product': 'smartphone', 'issue_type': 'battery', 'urgency': 'high', 'requested_action': 'repair'}
Base model: {'product': 'smartphone', 'issue_type': 'battery issue', 'urgency': 'high', 'requested_action': 'repair'}
LoRA model: {'product': 'smartphone', 'issue_type': 'battery', 'urgency': 'high', 'requested_action': 'repair'}

Example 3
Input: Could you classify this su

In [26]:
from collections import Counter

error_counts = Counter()

for item in failures:
    expected = item["expected"]
    predicted = item["parsed"]

    for field in expected:
        if predicted.get(field) != expected[field]:
            error_counts[field] += 1

print("Errors by field in the base model:")
for field, count in error_counts.items():
    print(f"{field}: {count}")

print("\nFine-tuned model errors:")
lora_errors = [
    item for item in lora_predictions
    if item["parsed"] != item["expected"]
]
print(len(lora_errors))

Errors by field in the base model:
issue_type: 15

Fine-tuned model errors:
0


In [27]:
adapter_path = "./qwen2.5-0.5b-lora-adapter"

trainer.model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

print("LoRA adapter saved successfully!")
print("Location:", adapter_path)

LoRA adapter saved successfully!
Location: ./qwen2.5-0.5b-lora-adapter
